In [1]:
pip install pandas

In [9]:
import os

folder_path = r"C:\Users\annam\OneDrive\Documents\GSE114725_rna_raw.csv"

print(os.listdir(folder_path))

['raw_corrected.csv']


In [17]:
import pandas as pd

file_path = r"C:\Users\annam\OneDrive\Documents\GSE114725_rna_raw.csv\raw_corrected.csv"

df_head = pd.read_csv(file_path, nrows=5)

print(df_head.columns[:10])


Index(['patient', 'tissue', 'replicate', 'cluster', 'cellid', 'A1BG', 'A2M',
       'A4GALT', 'AAAS', 'AACS'],
      dtype='object')


In [18]:
import pandas as pd
from scipy import sparse

file_path = r"C:\Users\annam\OneDrive\Documents\GSE114725_rna_raw.csv\raw_corrected.csv"

meta_cols = ["patient", "tissue", "replicate", "cluster", "cellid"]

# load metadata only
obs = pd.read_csv(file_path, usecols=meta_cols)

# load expression values in chunks
chunks = []

for chunk in pd.read_csv(file_path, chunksize=1000):
    expr = chunk.drop(columns=meta_cols)
    chunks.append(sparse.csr_matrix(expr.values))

X = sparse.vstack(chunks)

print(X.shape)
print(obs.shape)

(47016, 14875)
(47016, 5)


In [20]:
!pip install scanpy

   ---------------------------------------- 0.0/2.1 MB ? eta -:--:--
   -------------- ------------------------- 0.8/2.1 MB 5.6 MB/s eta 0:00:01
   ----------------------------- ---------- 1.6/2.1 MB 3.8 MB/s eta 0:00:01
   ---------------------------------------- 2.1/2.1 MB 3.9 MB/s eta 0:00:00
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   --- ------------------------------------ 1.0/11.0 MB 7.0 MB/s eta 0:00:02
   ------- -------------------------------- 2.1/11.0 MB 5.8 MB/s eta 0:00:02
   ----------- ---------------------------- 3.1/11.0 MB 5.4 MB/s eta 0:00:02
   ------------- -------------------------- 3.7/11.0 MB 4.5 MB/s eta 0:00:02
   ----------------- ---------------------- 4.7/11.0 MB 4.7 MB/s eta 0:00:02
   -------------------- ------------------- 5.8/11.0 MB 4.7 MB/s eta 0:00:02
   ------------------------ --------------- 6.8/11.0 MB 4.8 MB/s eta 0:00:01
   ---------------------------- ----------- 7.9/11.0 MB 4.7 MB/s eta 0:00:01
   --------------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.45.1 requires packaging<25,>=20, but you have packaging 26.2 which is incompatible.


In [21]:
import scanpy as sc

adata = sc.AnnData(X)

# add metadata
adata.obs = obs

# set proper cell IDs
adata.obs.index = obs["cellid"].astype(str)

# add gene names
adata.var_names = expr.columns

print(adata)

AnnData object with n_obs × n_vars = 47016 × 14875
    obs: 'patient', 'tissue', 'replicate', 'cluster', 'cellid'


C:\Users\annam\anaconda3\Lib\functools.py:934: ImplicitModificationWarning: Transforming to str index.
  return dispatch(args[0].__class__)(*args, **kw)


In [23]:
# make cell IDs the index
adata.obs.index = obs["cellid"].astype(str)

# remove duplicate cellid column from metadata
adata.obs = adata.obs.drop(columns=["cellid"])

# optional: give index a safe name
adata.obs.index.name = "cell_id"

# save
adata.write_h5ad("GSE114725_raw.h5ad")

In [24]:
print(adata)
print(adata.obs.head())

AnnData object with n_obs × n_vars = 47016 × 14875
    obs: 'patient', 'tissue', 'replicate', 'cluster'
        patient tissue  replicate  cluster
cell_id                                   
246         BC5  TUMOR          1        2
260         BC5  TUMOR          1        2
346         BC5  TUMOR          1        2
188         BC5  TUMOR          1        4
33          BC5  TUMOR          1        5


In [25]:
adata.write_h5ad("GSE114725_raw.h5ad")

In [26]:
adata_test = sc.read_h5ad("GSE114725_raw.h5ad")
print(adata_test)

AnnData object with n_obs × n_vars = 47016 × 14875
    obs: 'patient', 'tissue', 'replicate', 'cluster'


In [27]:
import os

print(os.getcwd())

C:\Users\annam


In [29]:
adata.write_h5ad(r"C:\Users\annam\OneDrive\Documents\Dissertation\GSE114725_raw.h5ad")

In [1]:
# Make sure observation and gene names are unique
adata.obs_names_make_unique()
adata.var_names_make_unique()

# Check metadata
adata.obs.head()

NameError: name 'adata' is not defined

In [2]:
import os
print(os.getcwd())

C:\Users\annam\Dissertation 2026


In [ ]:
print(adata)
print(adata.obs["patient"].value_counts())
print(adata.obs["tissue"].value_counts())
print(adata.obs["cluster"].value_counts())